# ML-05 — Feature Vector and Leakage/Privacy Check

The original assignment uses the FlyRank Hugging Face warehouse dataset. Due to repeated access issues with the gated dataset, I completed this notebook using the dataset provided inside the internship repository:

`../../data/raw/content_refresh_anonymized.csv`

This CSV contains the same type of anonymized search performance data required for learning the concepts of data contracts, feature engineering, and data leakage. All queries, feature engineering, and verification steps in this notebook are therefore performed on the repository CSV instead of the Hugging Face warehouse tables.


## 1. Build the feature vector

The feature vector uses five decision-time signals:

1. Search volume — represents the amount of search demand for the content.
2. CTR — represents how often impressions result in clicks.
3. Position quality — derived from average search position; a better position gives a higher value.
4. Content age — represents how old the content is.
5. Days since last update — represents content freshness/staleness.

The features are kept small and are intended to describe information that can be known before making a content refresh decision.

In [1]:
import pandas as pd
import numpy as np

# Load the repository CSV dataset.
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Build the five-feature vector for the Refresh / Content Opportunity lane.
feature_frame = df[
    [
        "search_volume",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update"
    ]
].copy()

# Make sure numeric columns are numeric.
numeric_features = [
    "search_volume",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

for col in numeric_features:
    feature_frame[col] = pd.to_numeric(
        feature_frame[col],
        errors="coerce"
    )

# Fill missing numeric values with the median of each feature.
for col in numeric_features:
    feature_frame[col] = feature_frame[col].fillna(
        feature_frame[col].median()
    )

# Engineer two simple representations.
feature_frame["log_search_volume"] = np.log1p(
    feature_frame["search_volume"]
)

feature_frame["position_quality"] = 1 / (
    1 + feature_frame["avg_position"]
)

# Keep exactly five final features.
feature_frame = feature_frame[
    [
        "log_search_volume",
        "ctr",
        "position_quality",
        "content_age_days",
        "days_since_last_update"
    ]
]

print("Feature vector shape:", feature_frame.shape)

display(feature_frame.head(10))

Feature vector shape: (30000, 5)


,log_search_volume,ctr,position_quality,content_age_days,days_since_last_update
0,2.397895,0.76,0.086207,187,20
1,4.510860,0.05,0.046948,445,25
2,0.000000,0.09,0.026667,141,20
3,2.397895,0.49,0.138889,463,22
4,0.000000,0.13,0.022222,263,14
5,6.580639,0.03,0.105263,147,20
6,0.000000,0.00,0.125000,90,20
7,6.381816,0.06,0.045045,445,22
8,0.000000,0.09,0.021277,90,20
9,0.000000,0.16,0.169492,257,104


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing values | Available when? |
|---|---|---|---|
| `log_search_volume` | Log-scaled search demand for the content. | Filled with the median. | Available before the decision because search demand has already been measured. |
| `ctr` | Click-through rate from observed impressions. | Filled with the median. | Available before the decision because it describes already observed performance. |
| `position_quality` | Transformed average search position where a higher value represents a better position. | `avg_position` is filled with the median before transformation. | Available before the decision because the current search position is already observed. |
| `content_age_days` | Number of days since the content was created. | Filled with the median. | Available before the decision because content age is already known. |
| `days_since_last_update` | Number of days since the content was last updated. | Filled with the median. | Available before the decision because update history is already known. |

No categorical feature is included in this first feature vector. The goal is to keep the initial vector small and focused on decision-time numeric signals.

In [2]:
feature_notes = pd.DataFrame({
    "feature": [
        "log_search_volume",
        "ctr",
        "position_quality",
        "content_age_days",
        "days_since_last_update"
    ],
    "meaning": [
        "Log-scaled search demand for the content.",
        "Click-through rate from observed impressions.",
        "Transformed average search position; higher means a better position.",
        "Age of the content in days.",
        "Days since the content was last updated."
    ],
    "missing_handling": [
        "Missing values replaced with the median.",
        "Missing values replaced with the median.",
        "Missing avg_position values replaced with the median before transformation.",
        "Missing values replaced with the median.",
        "Missing values replaced with the median."
    ],
    "available_when": [
        "Before the refresh decision because search demand is already observed.",
        "Before the refresh decision because CTR describes observed performance.",
        "Before the refresh decision because search position is already observed.",
        "Before the refresh decision because content age is known.",
        "Before the refresh decision because update history is known."
    ]
})

display(feature_notes)

,feature,meaning,missing_handling,available_when
0,log_search_volume,Log-scaled search demand for the content.,Missing values replaced with the median.,Before the refresh decision because search dem...
1,ctr,Click-through rate from observed impressions.,Missing values replaced with the median.,Before the refresh decision because CTR descri...
2,position_quality,Transformed average search position; higher me...,Missing avg_position values replaced with the ...,Before the refresh decision because search pos...
3,content_age_days,Age of the content in days.,Missing values replaced with the median.,Before the refresh decision because content ag...
4,days_since_last_update,Days since the content was last updated.,Missing values replaced with the median.,Before the refresh decision because update his...


## 3. The leakage hunt

I deliberately test a label-derived feature to demonstrate leakage.

For this experiment, I create a simple binary proxy label from CTR. I then intentionally copy that label into the feature frame as `leak_label_copy`.

This is deliberately wrong: the model is being given the answer it is supposed to predict.

I compare the score with the leaked feature against the score after removing it.

The leaked result is only a leakage demonstration. The final feature vector does not contain the label-derived column.

I also avoid using `trend_direction` and `trend_pct` as final model features because they are derived trend signals and could encode outcome information that would not be appropriate for an honest decision-time feature vector.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create a simple proxy label for the leakage demonstration.
# This is only for demonstrating leakage and is NOT a production label.
ctr_values = pd.to_numeric(df["ctr"], errors="coerce")

ctr_median = ctr_values.median()

label = (
    ctr_values
    .fillna(ctr_median)
    >= ctr_median
).astype(int)

print("Proxy label distribution:")
print(label.value_counts())

# DELIBERATE LEAKAGE
leaky_features = feature_frame.copy()

# Intentionally copy the label into the features.
# This is the leakage trap.
leaky_features["leak_label_copy"] = label.values

X_train, X_test, y_train, y_test = train_test_split(
    leaky_features,
    label,
    test_size=0.2,
    random_state=42,
    stratify=label
)

leaky_model = LogisticRegression(max_iter=1000)

leaky_model.fit(X_train, y_train)

leaky_predictions = leaky_model.predict(X_test)

leaky_accuracy = accuracy_score(
    y_test,
    leaky_predictions
)

print(
    "Accuracy with deliberate leakage:",
    round(leaky_accuracy, 4)
)

# REMOVE LEAKAGE
honest_features = feature_frame.copy()

X_train, X_test, y_train, y_test = train_test_split(
    honest_features,
    label,
    test_size=0.2,
    random_state=42,
    stratify=label
)

honest_model = LogisticRegression(max_iter=1000)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_predictions
)

print(
    "Honest accuracy after removing leakage:",
    round(honest_accuracy, 4)
)

print("\nLeakage lesson:")
print(
    "The label-derived feature was removed because it gives "
    "the model direct access to the answer."
)

Proxy label distribution:
ctr
1    15190
0    14810
Name: count, dtype: int64
Accuracy with deliberate leakage: 1.0
Honest accuracy after removing leakage: 0.9902

Leakage lesson:
The label-derived feature was removed because it gives the model direct access to the answer.


## 4. What I excluded and why

I deliberately excluded the following fields from the final feature vector:

- `trend_direction` — excluded because it is a derived trend signal and may contain outcome-related information.
- `trend_pct` — excluded because it is a derived trend measure and may encode information about the outcome.
- `leak_label_copy` — excluded because it directly copies the target and is a clear example of label leakage.
- `clicks_90d` — excluded to keep the first feature vector focused on a small set of decision-time signals.
- `clicks_last_30d` — excluded because the first feature vector does not need a direct click-count outcome measure.
- `clicks_prev_30d` — excluded because it is a period-comparison measure and is not required for this first feature vector.
- `sessions_90d` — excluded because it is an engagement outcome rather than one of the selected core refresh signals.
- `sessions_last_30d` — excluded because it is a recent engagement outcome and is not required for the initial vector.
- `sessions_prev_30d` — excluded because it is a period-comparison measure and is not required for the initial vector.

The final feature vector contains only five features and does not contain the deliberately leaked label.

In [4]:
excluded_features = pd.DataFrame({
    "field": [
        "trend_direction",
        "trend_pct",
        "leak_label_copy",
        "clicks_90d",
        "clicks_last_30d",
        "clicks_prev_30d",
        "sessions_90d",
        "sessions_last_30d",
        "sessions_prev_30d"
    ],
    "reason": [
        "Derived trend signal that may contain outcome-related information.",
        "Derived trend measure that may encode outcome information.",
        "Direct copy of the target label; clear leakage.",
        "Direct click-count outcome measure excluded from the small initial vector.",
        "Recent click outcome measure excluded from the initial vector.",
        "Period-comparison measure not required for the first vector.",
        "Engagement outcome excluded from the core refresh signals.",
        "Recent engagement outcome excluded from the initial vector.",
        "Period-comparison measure not required for the initial vector."
    ]
})

display(excluded_features)

print("\nFinal feature columns:")
print(feature_frame.columns.tolist())

,field,reason
0,trend_direction,Derived trend signal that may contain outcome-...
1,trend_pct,Derived trend measure that may encode outcome ...
2,leak_label_copy,Direct copy of the target label; clear leakage.
3,clicks_90d,Direct click-count outcome measure excluded fr...
4,clicks_last_30d,Recent click outcome measure excluded from the...
5,clicks_prev_30d,Period-comparison measure not required for the...
6,sessions_90d,Engagement outcome excluded from the core refr...
7,sessions_last_30d,Recent engagement outcome excluded from the in...
8,sessions_prev_30d,Period-comparison measure not required for the...



Final feature columns:
['log_search_volume', 'ctr', 'position_quality', 'content_age_days', 'days_since_last_update']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.